<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/crewai_support_triage_demo_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build Agents with CrewAI — Hands-On Lab
## Use case: **Automated Customer-Support Ticket Triage**

Welcome! This notebook is the practical companion to the slide deck
*"Build Agents with CrewAI: Agent Setup, Architecture, Roles, Autonomy & Focus,
Safety Guardrails, Applications."*

---

### The business problem (in one line)
> A support inbox receives messy, free-text customer emails. A human must read each one,
> classify it, look up the order, check the refund policy, and write a reply.
> **We will hand that whole job to a team of three AI agents.**

---

### What we are going to build

| # | Piece | Name in the code | Plain-English job |
|---|-------|------------------|-------------------|
| 1 | Tool | `lookup_order` | Reads the order database |
| 2 | Tool | `check_refund_policy` | Reads the company refund rulebook |
| 3 | Tool | `create_escalation` | Files a ticket for a human manager |
| 4 | Agent | `classifier_agent` | Reads the email, decides *what kind* of problem it is |
| 5 | Agent | `research_agent` | Gathers facts using tools 1 & 2 |
| 6 | Agent | `writer_agent` | Writes the customer reply, escalates if needed |
| 7 | Guardrail | `refund_guardrail` | Blocks any reply that promises too much money |
| 8 | Crew | `triage_crew` | Runs 4→5→6 in a fixed order (`Process.sequential`) |
| 9 | Agent | `triage_manager` | A **manager** that routes the same work dynamically (`Process.hierarchical`, Part 6) |

---

### How to read this notebook (important for non-coders)

Every code cell below is wrapped in this pattern:

1. A **markdown box like this one** tells you: *What this cell does*, *Input*, *Output*.
2. Inside the code, a **banner comment** repeats the purpose.
3. **Every imported package has a comment** explaining what that package is for.
4. **Every block of code has a comment** in plain English above it.

You can run the notebook top to bottom with `Shift + Enter`. Nothing is skipped.

---

### Prerequisites
* Python 3.10 or newer.
* A **free Hugging Face token** from <https://huggingface.co/settings/tokens> (type: *Read*).
  A plain free account is enough — no cloud console, no billing setup.
  Hugging Face is used here **only to reach the LLM** — it routes our request to whoever is
  hosting the Qwen weights today. Everything else (agents, tools, memory, guardrails) is CrewAI.

---
# PART 1 — Setup
---

### Cell 1 — Install the libraries

**What this cell does:** downloads the two Python packages this notebook needs.
Run it **once** per machine (or once per fresh Google Colab session). If you have already
installed them, running it again is harmless — pip will just say "already satisfied".

**Input:** nothing (it reads from the internet).

**Output:** a few lines of installation logs, ending with something like
`Successfully installed crewai-x.y.z litellm-... ...`. If you see a red "ERROR", scroll up: it is
almost always a Python-version problem (CrewAI needs Python ≥ 3.10).

ℹ️ **Why the `[litellm]` part?** You do not actually need it here: the Hugging Face router
speaks the OpenAI API dialect, which CrewAI supports natively. We install it anyway because it
costs nothing and it is what lets you swap in almost any other provider later without hitting
*"the LiteLLM fallback package is not installed."*

In [ ]:
# =============================================================================
# CELL 1 — INSTALL THE LIBRARIES
# -----------------------------------------------------------------------------
# "%pip install" is a Jupyter magic command. It installs a package into the SAME
# Python that is running this notebook (safer than plain "!pip install").
# The "-q" flag means "quiet" -> print fewer log lines.
#
# WHAT EACH PACKAGE IS FOR:
#   crewai            -> the multi-agent framework itself. Gives us Agent, Task,
#                        Crew, Process and LLM. This is the star of the show.
#   crewai[litellm]   -> the square brackets add an OPTIONAL EXTRA. LiteLLM is the
#                        universal adapter for ~100 model providers. We do NOT need it
#                        here (the Hugging Face router speaks the OpenAI dialect, which
#                        CrewAI supports out of the box) -- but it costs nothing and
#                        lets you swap in other providers later without surprises.
#   pydantic          -> a "data shape checker". We use it to say "the final answer
#                        MUST have these exact fields, with these exact types". This
#                        is what turns a chatty paragraph into a reliable,
#                        machine-readable record.
# =============================================================================

%pip install -q "crewai[litellm]" pydantic

# (Optional, not needed for this lab: `%pip install -q crewai-tools` gives you
#  ~40 ready-made tools such as web search and file readers. We are hand-building
#  our own tools instead, because that teaches the concept better.)

print("Install step finished. Now RESTART THE KERNEL if this was your first install.")

Install step finished. Now RESTART THE KERNEL if this was your first install.


### Cell 2 — Import everything we need

**What this cell does:** loads the tools of the trade into memory. Think of `import` as
*"open this toolbox and put it on my workbench"*. Nothing runs yet — we are only
laying out tools.

**Input:** nothing.

**Output:** a printed checklist confirming every import worked, e.g.
`✅ All imports OK — CrewAI is ready.` If an import fails, the cell stops with
`ModuleNotFoundError` and you should re-run Cell 1.

In [ ]:
# =============================================================================
# CELL 2 — IMPORTS  (open the toolboxes)
# -----------------------------------------------------------------------------
# Each line below is annotated with WHAT the package/class does for US.
# =============================================================================

# ---- Python's own built-in toolboxes (nothing to install) -------------------
import os                     # os  -> talks to the operating system. We use it ONLY to
                              #        store the Hugging Face token in an "environment variable"
                              #        so the key never has to be typed inside the agent code.
import json                   # json -> converts Python dictionaries <-> text. Agents talk in
                              #        text, our program thinks in dictionaries; json bridges them.
import re                     # re  -> regular expressions (pattern matching in text). We use it
                              #        once, to strip a reasoning model's <think>...</think> block
                              #        out of an answer before parsing the JSON inside it.
import textwrap               # textwrap -> neatly wraps long paragraphs when printing, so the
                              #        notebook output stays readable instead of one giant line.
from getpass import getpass   # getpass -> asks you to type a secret WITHOUT showing it on screen.
                              #        Used so your API key is not saved into the notebook file.
from typing import Tuple, Any # typing -> lets us label what a function returns. Purely for
                              #        human readability; Python does not enforce it.
from types import SimpleNamespace  # SimpleNamespace -> a throw-away "fake object". We use it
                              #        at the very end to test a guardrail without calling the LLM.

# ---- Pydantic: the data-shape checker ---------------------------------------
from pydantic import BaseModel, Field
# BaseModel -> the base class we inherit from to define "the answer must look like THIS".
# Field     -> lets us attach a human description to each field. The LLM READS these
#              descriptions, so good descriptions = better answers. This is real prompt design.

# ---- CrewAI: the multi-agent framework --------------------------------------
from crewai import Agent, Task, Crew, Process, LLM
# Agent   -> a worker with a role, a goal, a backstory and (optionally) tools.
# Task    -> one unit of work handed to an agent: a description + an expected_output.
# Crew    -> the team/manager object that owns the agents + tasks and actually runs them.
# Process -> an enum choosing HOW the crew runs: Process.sequential or Process.hierarchical.
# LLM     -> CrewAI's wrapper around any language model provider. We point it at the
#            Hugging Face router, which forwards to a Qwen model.

from crewai.tools import tool
# tool -> a "decorator". Writing @tool above a normal Python function turns that plain
#         function into something an agent is allowed to call by itself at runtime.

# TaskOutput is the object CrewAI hands to a guardrail function. Its import path has
# moved between versions, so we try both. (This try/except is defensive, not magic.)
try:
    from crewai.tasks.task_output import TaskOutput
except ImportError:                       # pragma: no cover
    from crewai import TaskOutput

print("All imports OK — CrewAI is ready.")

All imports OK — CrewAI is ready.


### Cell 3 — Connect to Hugging Face (the gateway to the brain)

**What this cell does:** asks you for your **Hugging Face access token** and stores it in an
environment variable named `HF_TOKEN`.

**Where the token comes from:** <https://huggingface.co/settings/tokens> → *New token* → type
**Read** is enough. It looks like `hf_xxxxxxxx…`. A plain free Hugging Face account is all you
need — no cloud console, no billing setup, no separate provider signup. That is exactly why we
are using it here.

**What Hugging Face actually is in this architecture — this matters:** Hugging Face is not the
model. It is a **router**. Its Inference Providers endpoint speaks the OpenAI API dialect and
forwards your request to whichever hosting company actually runs the weights (Together, Novita,
Cerebras, Groq, Fireworks, and others). So one token and one URL give you access to hundreds of
open models — including **Qwen** — without holding an account with any of them.

That is a genuinely useful idea for the class: *the LLM is a swappable component behind a URL.*
Everything else in this notebook — agents, tools, guardrails — stays byte-for-byte identical
whichever model comes back.

**Free tier:** Hugging Face includes a monthly credit allowance for Inference Providers, larger
for PRO accounts. It is enough for this lab, but it is **not unlimited** — so Cell 4 deliberately
picks the cheapest route and caps every reply. Current amounts:
<https://huggingface.co/docs/inference-providers/pricing>.

**Input:** you type/paste your token into the prompt box that appears. It is hidden as you type.

**Output:** `Hugging Face token stored (length = 37, starts with 'hf_').`

In [ ]:
# =============================================================================
# CELL 3 — STORE THE HUGGING FACE TOKEN
# -----------------------------------------------------------------------------
# An "environment variable" is a named value that lives in the running process.
# Libraries look it up by name. This keeps the secret OUT of your source code, so
# you can safely share this notebook with classmates.
#
# Get a free token at:  https://huggingface.co/settings/tokens   (a "Read" token is enough)
# =============================================================================

# The OpenAI-compatible gateway. Every request we make goes here, and Hugging Face
# forwards it to whichever company is actually hosting the model we asked for.
HF_ROUTER = "https://router.huggingface.co/v1"

# Only ask for the token if it is not already sitting in the environment.
# (This means re-running the cell will not pester you a second time.)
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass(
        "Paste your Hugging Face token (hf_...) - input is hidden: ").strip()

token = os.environ["HF_TOKEN"]

# Print PROOF that a token exists without ever printing the token itself.
print(f"Hugging Face token stored (length = {len(token)}, starts with '{token[:3]}').")

# A tiny sanity check. HF tokens begin with "hf_". If yours does not, you have probably
# copied something else - a model URL, or a token from a different site.
if not token.startswith("hf_"):
    print("WARNING: Hugging Face tokens normally start with 'hf_'. "
          "Check you copied an Access Token from huggingface.co/settings/tokens.")

print(f"Router endpoint: {HF_ROUTER}")

# NOTE FOR THE CLASS: if you are on Google Colab you may prefer the Secrets panel:
#   from google.colab import userdata
#   os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

Paste your Hugging Face token (hf_...) - input is hidden: ··········
Hugging Face token stored (length = 37, starts with 'hf_').
Router endpoint: https://router.huggingface.co/v1


### Cell 4 — Ask Hugging Face which models are live, pick a Qwen, build the `LLM`, smoke-test it

**What this cell does:** four things, in order.

1. **Asks the router which models are actually served right now**, via `GET /v1/models`. Which
   open models are hosted changes week to week as providers add and drop them, so a hardcoded
   name is a liability.
2. Filters that list to **Qwen instruct/chat** models, dropping vision, audio and embedding
   variants — they cannot drive a text agent.
3. Picks the first from our preference list that is actually present, and appends the
   **`:cheapest`** routing policy.
4. Builds one `LLM` object and sends a throw-away message to prove the connection works.

---

#### Two naming details that confuse everyone

**(a) The model string starts with `openai/` — and has nothing to do with OpenAI.**

```python
LLM(model="openai/Qwen/Qwen3-32B:cheapest", base_url=HF_ROUTER, api_key=HF_TOKEN)
```

CrewAI uses the first path segment to choose a *client*, not a company. `openai/` means
*"talk the OpenAI API dialect"*; `base_url` then points that client at Hugging Face instead of
at OpenAI. So the string reads: **OpenAI dialect → Hugging Face router → Qwen weights.** CrewAI
strips the `openai/` prefix and sends `Qwen/Qwen3-32B:cheapest` as the model name, which is
exactly what the router expects.

**(b) The `:cheapest` suffix is a routing policy, not part of the model name.**

Hugging Face model IDs take an optional policy suffix that decides *which host* serves you:

| Suffix | Meaning |
|---|---|
| `:cheapest` | lowest price per output token — **what we use, to protect your free credits** |
| `:fastest` | highest throughput (the default if you write no suffix) |
| `:preferred` | your own provider order, set in HF settings |
| `:together`, `:novita`, `:groq`, … | pin one specific hosting company |

Same weights either way — you are choosing a landlord, not a model.

---

**Key setting — `temperature=0.1`:** temperature controls randomness (0 = very predictable,
1 = very creative). For **support triage we want boring and repeatable**, so we keep it low.
This is your first taste of *"Focus"* from the deck: we deliberately narrow the model's freedom.

**Input:** the `HF_TOKEN` and `HF_ROUTER` from Cell 3.

**Output:**
```
Discovered 312 models on the router; 9 of them are Qwen text models.
Candidates (first 10):
    Qwen/Qwen2.5-72B-Instruct
    Qwen/Qwen3-235B-A22B-Instruct-2507
    Qwen/Qwen3-32B
    Qwen/Qwen3-8B
    ...
Selected  -> Qwen/Qwen3-32B:cheapest
LLM built -> Qwen/Qwen3-32B:cheapest  @ https://router.huggingface.co/v1
Smoke test reply: READY
```

---

#### ⚠️ Troubleshooting

| Symptom | What it actually means |
|---|---|
| `401 Unauthorized` | Token wrong, expired, or you pasted something that is not an `hf_…` token. |
| `402 Payment Required` / *monthly credits exceeded* | Free allowance used up for the month. Switch to a smaller model with `use_model(...)`, or wait for the reset. |
| `Model not supported` / `404` | No provider is currently serving that model. Re-run this cell — the printed list is the only current answer. |
| The reply is empty, or the guardrail rejects everything | You picked a **reasoning** model (`-Thinking`, `QwQ`). It emits `<think>…</think>` before the answer. Our guardrail strips that (Cell 13), but a non-reasoning instruct model is simpler — the preference list below avoids them. |
| The error names a model you no longer use | Stale object. You edited the model but did not re-run Cells 10–12 and 15 — they hold the `llm` they were built with. *Kernel → Restart & Run All*. |

**The lesson for the class:** a model ID is a *dependency with an expiry date*, not a constant.
Discover it, validate it, and let your code fall back when it disappears. That is what Steps 1–4
do in about twenty lines.

In [ ]:
# =============================================================================
# CELL 4 — LIVE MODEL DISCOVERY, THEN BUILD THE SHARED LLM
# -----------------------------------------------------------------------------
# Which open models are hosted changes constantly. Instead of trusting a hardcoded
# string, we ASK the router what is served today. This cell never guesses silently.
# =============================================================================

import urllib.request   # standard library: makes one plain HTTPS GET request.
                        # We use it (instead of `requests`) so there is nothing extra to install.
import urllib.error     # standard library: the specific error types urllib raises, so we can
                        #        catch "401 / 404" politely instead of crashing the notebook.

# --- STEP 0 — clear stale overrides -----------------------------------------
# Other tutorials often leave these environment variables behind, and CrewAI will
# silently prefer them over your variable. This is a very common cause of
# "I changed the model but it still uses the old one".
for stale in ("OPENAI_MODEL_NAME", "MODEL", "LITELLM_MODEL", "OPENAI_BASE_URL"):
    if os.environ.pop(stale, None):
        print(f"Cleared stale environment variable: {stale}")


# --- STEP 1 — ask the router which models are served right now ---------------
def list_hf_models():
    """Return every model id the Hugging Face router currently serves, or None.

    This hits the OpenAI-compatible GET /v1/models endpoint. If it is unavailable
    (network, proxy, token problem) we fall back to a curated list rather than crash.
    """
    request = urllib.request.Request(
        f"{HF_ROUTER.rstrip('/')}/models",
        headers={"Authorization": f"Bearer {os.environ['HF_TOKEN']}"},
    )
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            payload = json.loads(response.read().decode())
        # The response shape is {"data": [{"id": "Qwen/Qwen3-32B", ...}, ...]}
        return sorted(entry["id"] for entry in payload.get("data", []))
    except urllib.error.HTTPError as e:
        print(f"(Could not list models: HTTP {e.code}. Falling back to a curated list.)")
        if e.code == 401:
            print("  A 401 means the token was rejected - re-check Cell 3.")
        return None
    except Exception as e:                       # network hiccup, proxy, DNS, timeout
        print(f"(Could not list models: {type(e).__name__}. Falling back to a curated list.)")
        return None


# A safety net used only when discovery is unavailable.
FALLBACK_MODELS = [
    "Qwen/Qwen3-32B",
    "Qwen/Qwen2.5-72B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
]

discovered = list_hf_models()

# --- STEP 2 — keep only the Qwen TEXT models ---------------------------------
# We want chat/instruct models. Vision (-VL), audio, embedding and reranker variants
# share the Qwen name but cannot drive a text agent.
NOT_TEXT = ("-vl", "-audio", "-omni", "embedding", "rerank", "-asr", "-tts",
            "image", "-math", "-coder-", "guard")

if discovered:
    QWEN_MODELS = [m for m in discovered
                   if m.lower().startswith("qwen/")
                   and not any(word in m.lower() for word in NOT_TEXT)]
    print(f"Discovered {len(discovered)} models on the router; "
          f"{len(QWEN_MODELS)} of them are Qwen text models.")
    if not QWEN_MODELS:                    # router is up but serving no Qwen today
        QWEN_MODELS = FALLBACK_MODELS
        print("No Qwen models listed right now - using the fallback list.")
else:
    QWEN_MODELS = FALLBACK_MODELS
    print(f"Using {len(QWEN_MODELS)} fallback model names.")

print("Candidates (first 10):")
for m in QWEN_MODELS[:10]:
    print("   ", m)
if len(QWEN_MODELS) > 10:
    print(f"    ... and {len(QWEN_MODELS) - 10} more")


# --- STEP 3 — pick the first preferred model that is ACTUALLY served ---------
# Ordered best-first for THIS task (following instructions and emitting clean JSON).
# Deliberately NON-reasoning instruct models: a "-Thinking" or "QwQ" model wraps its
# answer in <think>...</think>, which is extra work for no benefit in ticket triage.
PREFERRED = [
    "Qwen/Qwen3-32B",                     # good instruction-following, mid-size
    "Qwen/Qwen2.5-72B-Instruct",          # larger, very reliable with JSON
    "Qwen/Qwen3-235B-A22B-Instruct-2507", # largest; costs more credits per run
    "Qwen/Qwen2.5-7B-Instruct",           # small + cheap; best for conserving credits
    "Qwen/Qwen3-8B",
]

# The routing POLICY, not part of the model name. ":cheapest" sends the request to the
# lowest-priced host serving these weights - the right default on a free allowance.
# Alternatives: ":fastest" (default), ":preferred", or pin one host e.g. ":together".
POLICY = ":cheapest"

chosen = next((m for m in PREFERRED if m in QWEN_MODELS), None)
if chosen is None:
    if not QWEN_MODELS:
        raise RuntimeError("No Qwen text models available. Check the token in Cell 3.")
    chosen = QWEN_MODELS[0]
    print(f"\nNone of the preferred models were listed; falling back to {chosen}.")

HF_MODEL = f"{chosen}{POLICY}"

# "openai/" is CrewAI's CLIENT selector, not a company: it means "speak the OpenAI
# dialect". base_url then aims that client at Hugging Face instead of at OpenAI.
CREWAI_MODEL = f"openai/{HF_MODEL}"
print(f"\nSelected  -> {HF_MODEL}")


# --- STEP 4 — a safe way to switch models later -------------------------------
def use_model(name: str, policy: str = POLICY) -> str:
    """Validate a model name against the LIVE list and return the CrewAI model string.

    Call it like:  CREWAI_MODEL = use_model("Qwen/Qwen2.5-7B-Instruct")
    It refuses names the router is not serving and suggests the nearest real one,
    instead of letting you discover the problem inside a confusing agent traceback.
    """
    import difflib   # standard library: finds the closest spelling among a list of words.

    # Strip anything the user may have pasted in from another provider's format.
    bare = name.replace("openai/", "", 1).split(":")[0].strip()

    if bare not in QWEN_MODELS:
        near = difflib.get_close_matches(bare, QWEN_MODELS, n=3, cutoff=0.4)
        hint = f" Did you mean: {', '.join(near)}?" if near else ""
        raise ValueError(f"'{bare}' is not currently served by the router.{hint}\n"
                         f"Available Qwen models: {QWEN_MODELS}")

    return f"openai/{bare}{policy}"


# --- STEP 5 — build the one LLM object every agent will share -----------------
#   base_url    -> point the OpenAI-dialect client at Hugging Face, not at OpenAI.
#   api_key     -> your hf_... token.
#   temperature = 0.1  -> low randomness. Same ticket in, near-identical answer out.
#   max_tokens  = 1200 -> a hard ceiling on reply length. This is a COST guardrail:
#                         it stops a runaway agent from writing a 20-page essay,
#                         and on a metered free allowance that matters immediately.
llm = LLM(
    model=CREWAI_MODEL,
    base_url=HF_ROUTER,
    api_key=os.environ["HF_TOKEN"],
    temperature=0.1,
    max_tokens=1200,
)

print(f"LLM built -> {llm.model}  @ {llm.base_url}")

# --- STEP 6 — smoke test ------------------------------------------------------
# Send one tiny message straight to the model, bypassing all agents. If this works,
# the token + router + model name are all correct, so any later failure is an AGENT bug.
smoke = llm.call([{"role": "user", "content": "Reply with the single word: READY"}])
print("Smoke test reply:", str(smoke).strip())

# REMINDER: if you change the model above, you must ALSO re-run the agent cells
# (10, 11, 12) and the crew cell (15). They captured the OLD llm object.

Discovered 162 models on the router; 18 of them are Qwen text models.
Candidates (first 10):
    Qwen/Qwen2.5-72B-Instruct
    Qwen/Qwen3-14B
    Qwen/Qwen3-235B-A22B-Instruct-2507
    Qwen/Qwen3-235B-A22B-Thinking-2507
    Qwen/Qwen3-30B-A3B
    Qwen/Qwen3-30B-A3B-Instruct-2507
    Qwen/Qwen3-32B
    Qwen/Qwen3-Next-80B-A3B-Instruct
    Qwen/Qwen3.5-122B-A10B
    Qwen/Qwen3.5-27B
    ... and 8 more

Selected  -> Qwen/Qwen3-32B:cheapest
LLM built -> Qwen/Qwen3-32B:cheapest  @ https://router.huggingface.co/v1


ERROR:root:OpenAI API call failed: Error code: 402 - {'error': 'Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.'}
ERROR:root:OpenAI API call failed: Error code: 402 - {'error': 'Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.'}


[CrewAIEventsBus] Warning: Ending event 'llm_call_failed' emitted with empty scope stack. Missing starting event?

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: OpenAI API call failed: Error code: 402 - {'error': 'Purchase pre-paid credits to continue using        │
│  Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.'}                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: OpenAI API call failed: Error code: 402 - {'error': 'Purchase pre-paid credits to continue using        │
│  Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.'}                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

APIStatusError: Error code: 402 - {'error': 'Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.'}

---
# PART 2 — The company's data and the TOOLS that read it

**Concept check — what is a "tool"?**
An LLM on its own can only *talk*. It cannot look up your order or file a ticket.
A **tool** is an ordinary Python function that we *grant* to an agent. At runtime the agent
decides — on its own — whether to call it, and with what arguments. That decision is the
"agency" in *agentic AI*.
---

### Cell 5 — The fake company database

**What this cell does:** creates two small Python dictionaries that stand in for real
company systems. In a real deployment `ORDERS` would be a SQL query and `REFUND_POLICY`
would be a document in a vector store — but the *agent code above them would not change
one line*. That is the point of this cell.

**Input:** nothing.

**Output:** `Mock systems loaded: 3 orders, 4 policy rules.`

In [ ]:
# =============================================================================
# CELL 5 — MOCK COMPANY SYSTEMS (stand-ins for real databases)
# -----------------------------------------------------------------------------
# We hard-code the data so the lab runs offline and gives identical results for
# every student. Swap these two dictionaries for a real DB call later — nothing
# else in the notebook has to change.
# =============================================================================

# --- The "order management system" -------------------------------------------
# Each key is an order ID. Each value describes what the customer bought.
ORDERS = {
    "A-1001": {
        "customer":       "Ravi Kumar",
        "item":           "Noise-cancelling headphones",
        "amount_usd":     180.00,
        "status":         "delivered",
        "delivered_days_ago": 5,
        "prior_refunds":  0,
    },
    "A-1002": {
        "customer":       "Meera Iyer",
        "item":           "4K monitor",
        "amount_usd":     650.00,
        "status":         "delivered",
        "delivered_days_ago": 40,
        "prior_refunds":  1,
    },
    "A-1003": {
        "customer":       "John Doe",
        "item":           "Mechanical keyboard",
        "amount_usd":     95.00,
        "status":         "in_transit",
        "delivered_days_ago": None,
        "prior_refunds":  0,
    },
}

# --- The "policy knowledge base" ---------------------------------------------
# Plain-English rules the research agent will quote back to us.
REFUND_POLICY = {
    "return_window_days": 30,
    "auto_refund_limit_usd": 200.00,
    "rules": [
        "Refunds are allowed only within 30 days of delivery.",
        "Refunds up to USD 200 can be approved automatically by support.",
        "Refunds above USD 200 REQUIRE a human manager's approval.",
        "Orders still in transit cannot be refunded; offer re-delivery or cancellation.",
    ],
}

print(f"Mock systems loaded: {len(ORDERS)} orders, {len(REFUND_POLICY['rules'])} policy rules.")

Mock systems loaded: 3 orders, 4 policy rules.


### Cell 6 — TOOL 1: `lookup_order`

**What this cell does:** turns a normal Python function into an agent-callable tool using the
`@tool(...)` decorator.

**Two things to notice — they matter more than the code:**
1. **The docstring is the tool's advertisement.** The agent never sees your Python body; it
   only sees the tool *name* + *docstring* and uses them to decide when to call it. A vague
   docstring = an agent that calls the wrong tool.
2. **We return a string.** Agents read text, so we hand back a neat one-line summary.

**Input (at runtime, chosen by the agent):** an `order_id` string such as `"A-1001"`.

**Output (returned to the agent):** e.g.
`Order A-1001 | customer=Ravi Kumar | item=Noise-cancelling headphones | amount_usd=180.0 | status=delivered | delivered_days_ago=5 | prior_refunds=0`
or, for an unknown ID, `ERROR: order 'A-9999' not found...`.

**Output of running the cell itself:** `Tool registered: Order Lookup` plus a demo call so
you can see the exact text an agent would receive.

In [ ]:
# =============================================================================
# CELL 6 — TOOL 1: LOOK UP AN ORDER
# -----------------------------------------------------------------------------
# @tool("Order Lookup") is a DECORATOR. It wraps the plain function below and
# registers it with CrewAI so an agent can invoke it during its own reasoning loop.
# The text "Order Lookup" is the name the agent will see and say out loud in logs.
# =============================================================================

@tool("Order Lookup")
def lookup_order(order_id: str) -> str:
    """Look up one customer order by its order ID (for example 'A-1001').

    Use this whenever you need facts about a purchase: who bought it, the item,
    the amount paid in USD, the delivery status, how many days ago it was
    delivered, and how many refunds this customer already received.
    Returns a plain-text summary, or an ERROR line if the ID does not exist.
    """
    # STEP 1 — Clean the input. LLMs often pass ' a-1001 ' with stray spaces or
    #          lower case, so we normalise before looking anything up.
    key = order_id.strip().upper()

    # STEP 2 — Try to find the order. If it is missing we return a helpful ERROR
    #          string instead of raising an exception. Why? Because an agent can
    #          READ an error string and recover ("let me re-check the ID"), but a
    #          Python crash would kill the whole run.
    if key not in ORDERS:
        return (f"ERROR: order '{order_id}' not found. "
                f"Known order IDs are: {', '.join(ORDERS.keys())}.")

    # STEP 3 — Format the record as one readable line of text for the agent.
    o = ORDERS[key]
    return (
        f"Order {key} | customer={o['customer']} | item={o['item']} | "
        f"amount_usd={o['amount_usd']} | status={o['status']} | "
        f"delivered_days_ago={o['delivered_days_ago']} | prior_refunds={o['prior_refunds']}"
    )


print("Tool registered:", lookup_order.name)
# Demo call so you can SEE what the agent will receive.
# (.run() is how we humans call a CrewAI tool directly; agents call it themselves.)
print("Demo  ->", lookup_order.run(order_id="A-1001"))
print("Demo  ->", lookup_order.run(order_id="A-9999"))

Tool registered: Order Lookup
Demo  -> Order A-1001 | customer=Ravi Kumar | item=Noise-cancelling headphones | amount_usd=180.0 | status=delivered | delivered_days_ago=5 | prior_refunds=0
Demo  -> ERROR: order 'A-9999' not found. Known order IDs are: A-1001, A-1002, A-1003.


### Cell 7 — TOOL 2: `check_refund_policy`

**What this cell does:** gives the agent a way to read the rulebook. Notice that this tool
does **arithmetic** (`amount <= limit`) rather than leaving the maths to the LLM.

**This is a deliberate design rule from the deck:** *let the model decide, let the code
calculate.* LLMs are unreliable at arithmetic and policy edge cases; Python is perfect at it.
Every number you move out of the prompt and into a tool is one less way for the agent to be wrong.

**Input (at runtime):** `amount_usd` (a number) and `days_since_delivery` (a number).

**Output:** a verdict string such as
`POLICY VERDICT: AUTO-APPROVE. Amount 180.0 USD is within the 200.0 USD auto-approval limit and 5 days is inside the 30-day window.`

**Output of running the cell:** three demo verdicts covering the three branches.

In [ ]:
# =============================================================================
# CELL 7 — TOOL 2: CHECK THE REFUND POLICY
# -----------------------------------------------------------------------------
# DESIGN RULE: the LLM decides WHETHER to check policy; this Python code decides
# WHAT the policy says. Never ask a language model to be your rules engine.
# =============================================================================

@tool("Refund Policy Check")
def check_refund_policy(amount_usd: float, days_since_delivery: int) -> str:
    """Decide whether a refund of a given amount is allowed under company policy.

    Pass the order amount in USD and how many days ago the order was delivered.
    Returns one of three verdicts: AUTO-APPROVE, NEEDS-HUMAN-APPROVAL, or DENIED,
    each with the reason. Always call this before promising a refund to a customer.
    """
    # STEP 1 — Be forgiving about types. The agent may hand us "180" as text,
    #          so we convert to numbers ourselves rather than crashing.
    try:
        amount = float(amount_usd)
        days = int(days_since_delivery)
    except (TypeError, ValueError):
        return ("ERROR: amount_usd must be a number and days_since_delivery must be a "
                "whole number. Example: amount_usd=180, days_since_delivery=5")

    # STEP 2 — Pull the thresholds out of the policy dictionary (never hard-code twice).
    window = REFUND_POLICY["return_window_days"]
    limit = REFUND_POLICY["auto_refund_limit_usd"]

    # STEP 3 — Rule A: outside the return window -> refuse outright.
    if days > window:
        return (f"POLICY VERDICT: DENIED. The order was delivered {days} days ago, "
                f"which is outside the {window}-day return window. "
                f"Offer a repair, replacement or store credit instead of a refund.")

    # STEP 4 — Rule B: inside the window but expensive -> a human must sign off.
    if amount > limit:
        return (f"POLICY VERDICT: NEEDS-HUMAN-APPROVAL. Amount {amount} USD exceeds the "
                f"{limit} USD auto-approval limit. A manager must approve. "
                f"Do NOT promise the refund to the customer; promise a review instead.")

    # STEP 5 — Rule C: inside the window and cheap -> the agent may approve it alone.
    return (f"POLICY VERDICT: AUTO-APPROVE. Amount {amount} USD is within the {limit} USD "
            f"auto-approval limit and {days} days is inside the {window}-day window.")


print("Tool registered:", check_refund_policy.name)
print("Demo 1 ->", check_refund_policy.run(amount_usd=180, days_since_delivery=5))
print("Demo 2 ->", check_refund_policy.run(amount_usd=650, days_since_delivery=10))
print("Demo 3 ->", check_refund_policy.run(amount_usd=650, days_since_delivery=)40)

Tool registered: Refund Policy Check
Demo 1 -> POLICY VERDICT: AUTO-APPROVE. Amount 180.0 USD is within the 200.0 USD auto-approval limit and 5 days is inside the 30-day window.
Demo 2 -> POLICY VERDICT: NEEDS-HUMAN-APPROVAL. Amount 650.0 USD exceeds the 200.0 USD auto-approval limit. A manager must approve. Do NOT promise the refund to the customer; promise a review instead.
Demo 3 -> POLICY VERDICT: DENIED. The order was delivered 40 days ago, which is outside the 30-day return window. Offer a repair, replacement or store credit instead of a refund.


### Cell 8 — TOOL 3: `create_escalation` (a tool that *acts*)

**What this cell does:** tools 1 and 2 only **read**. This one **writes** — it creates a ticket
for a human manager. Any tool that changes the world (sends email, issues a refund, deletes a
row) deserves extra care, so we deliberately:

* keep an audit log (`ESCALATION_LOG`) of every call,
* require a `reason` argument so nothing is escalated silently,
* return a ticket ID so the agent can quote it to the customer.

**Input (at runtime):** `order_id`, `reason`, `amount_usd`.

**Output:** `ESCALATION CREATED: ticket ESC-001 for order A-1002 ...`

**Output of running the cell:** one demo escalation and the resulting audit log.

In [ ]:
# =============================================================================
# CELL 8 — TOOL 3: ESCALATE TO A HUMAN  (a WRITE / action tool)
# -----------------------------------------------------------------------------
# In production this would call Zendesk / ServiceNow / Jira. Here it appends to a
# Python list so you can inspect exactly what the agent chose to do.
# =============================================================================

# The audit trail. Everything the agent escalates lands here so a human can review it.
ESCALATION_LOG = []


@tool("Create Human Escalation")
def create_escalation(order_id: str, reason: str, amount_usd: float = 0.0) -> str:
    """Create a ticket asking a HUMAN manager to review this case.

    Use this tool whenever the refund policy says NEEDS-HUMAN-APPROVAL, whenever the
    customer is threatening legal action, or whenever you are unsure what to do.
    Escalating is always safer than guessing. Returns the new escalation ticket ID.
    """
    # STEP 1 — Build a ticket ID like ESC-001, ESC-002, ... based on how many exist.
    ticket_id = f"ESC-{len(ESCALATION_LOG) + 1:03d}"

    # STEP 2 — Record the ticket in our audit log. Note that we store the REASON,
    #          so later we can prove why the machine handed the case to a person.
    ESCALATION_LOG.append({
        "ticket_id": ticket_id,
        "order_id": order_id.strip().upper(),
        "reason": reason,
        "amount_usd": amount_usd,
    })

    # STEP 3 — Tell the agent what happened, in text it can quote to the customer.
    return (f"ESCALATION CREATED: ticket {ticket_id} for order {order_id} "
            f"(amount {amount_usd} USD). Reason: {reason}. "
            f"A human manager will respond within one business day.")


print("Tool registered:", create_escalation.name)
print("Demo ->", create_escalation.run(order_id="A-1002",
                                       reason="Refund above auto-approval limit",
                                       amount_usd=650))
print("Audit log now:", ESCALATION_LOG)

# Clear the demo entry so the real run below starts from a clean slate.
ESCALATION_LOG.clear()
print("Audit log cleared for the real run:", ESCALATION_LOG)

Tool registered: Create Human Escalation
Demo -> ESCALATION CREATED: ticket ESC-001 for order A-1002 (amount 650.0 USD). Reason: Refund above auto-approval limit. A human manager will respond within one business day.
Audit log now: [{'ticket_id': 'ESC-001', 'order_id': 'A-1002', 'reason': 'Refund above auto-approval limit', 'amount_usd': 650.0}]
Audit log cleared for the real run: []


---
# PART 3 — The answer's shape, the AGENTS, and the GUARDRAIL
---

### Cell 9 — Define the shape of the final answer (`TicketTriage`)

**What this cell does:** declares a Pydantic model. This is us saying to the agent:
*"do not send me an essay — send me exactly these eight fields."*

**Why this is one of the most important cells in the notebook:** free-form text cannot be
plugged into a CRM, a dashboard or an `if` statement. A typed object can. Structured output is
what turns a demo into a system.

Notice each `Field(..., description="...")`. **The LLM reads those descriptions.** They are
part of your prompt, not just documentation — so write them like instructions.

**Input:** nothing.

**Output:** `Schema defined with 8 fields: category, urgency, order_id, ...`

In [ ]:
# =============================================================================
# CELL 9 — THE OUTPUT CONTRACT (a Pydantic model)
# -----------------------------------------------------------------------------
# "class TicketTriage(BaseModel)" means: create a new data shape called TicketTriage
# that follows Pydantic's rules. Pydantic will REJECT an answer that is missing a
# field or that puts text where a number belongs.
# =============================================================================

class TicketTriage(BaseModel):
    """The one and only structure a finished triage may take."""

    # Each line reads: field_name : type = Field(default, description shown to the LLM)
    category: str = Field(
        ..., description="One of: refund, delivery, technical, billing, other")
    urgency: str = Field(
        ..., description="One of: low, medium, high")
    order_id: str = Field(
        ..., description="The order ID mentioned by the customer, e.g. A-1001. Use 'unknown' if absent.")
    policy_verdict: str = Field(
        ..., description="Exactly what the Refund Policy Check tool returned: AUTO-APPROVE, NEEDS-HUMAN-APPROVAL, DENIED, or NOT-APPLICABLE.")
    refund_amount_usd: float = Field(
        0.0, description="Refund amount you are promising in USD. Use 0 if you are not promising any refund.")
    needs_human_escalation: bool = Field(
        False, description="true if a human manager must review this case, otherwise false.")
    escalation_ticket: str = Field(
        "none", description="The ESC-xxx ticket ID if you escalated, otherwise the word 'none'.")
    customer_reply: str = Field(
        ..., description="The final email text to send to the customer. Polite, under 150 words, no promises the policy does not allow.")


# "..." (Ellipsis) as the first argument means the field is REQUIRED.
# A value like 0.0 or False means the field is optional with that default.
print(f"Schema defined with {len(TicketTriage.model_fields)} fields: "
      f"{', '.join(TicketTriage.model_fields.keys())}")

Schema defined with 8 fields: category, urgency, order_id, policy_verdict, refund_amount_usd, needs_human_escalation, escalation_ticket, customer_reply


### Cell 10 — AGENT 1: `classifier_agent` (the receptionist)

**What this cell does:** creates our first agent. An agent is defined by four ideas:

| Parameter | Plain English | Why it matters |
|---|---|---|
| `role` | its job title | the model role-plays this |
| `goal` | the single outcome it owns | keeps it **focused** — the "Focus" pillar |
| `backstory` | who it is / how it thinks | quietly sets tone and standards |
| `tools` | what it is allowed to touch | this is a **permission list**, i.e. a guardrail |

Our classifier gets **no tools at all** — reading an email needs no database. *Give an agent the
smallest toolset that lets it finish its job;* every extra tool is another way to go off-track.

`max_iter=3` caps how many think→act loops it may run — a cost and runaway guardrail.

**Input:** the shared `llm` from Cell 4.

**Output:** `Agent ready: Ticket Intake Classifier | tools: 0 | max_iter: 3`

In [ ]:
# =============================================================================
# CELL 10 — AGENT 1: THE CLASSIFIER  (reads the email, labels it)
# -----------------------------------------------------------------------------
# Think of role/goal/backstory as the agent's job advert, KPI and CV.
# =============================================================================

classifier_agent = Agent(
    # role -> the job title. The model literally acts this part.
    role="Ticket Intake Classifier",

    # goal -> ONE sentence describing the single outcome this agent owns.
    #         Narrow goals produce focused agents. Broad goals produce wanderers.
    goal=("Read a raw customer support email and label it accurately with a category, "
          "an urgency level, and the order ID it refers to."),

    # backstory -> context that shapes tone and judgement. It is still prompt text,
    #              so make it useful: here we tell it to prefer caution.
    backstory=("You have triaged support tickets for eight years at a consumer electronics "
               "retailer. You are fast, literal and never invent details. If the customer "
               "did not mention an order ID, you write 'unknown' rather than guessing. "
               "When a customer sounds angry or mentions legal action, you mark urgency high."),

    tools=[],          # NO TOOLS. Classification needs no database access.
                       # Least privilege: an agent cannot misuse a tool it was never given.

    llm=llm,           # the shared Qwen-via-Hugging-Face brain from Cell 4.

    verbose=True,      # print the agent's thinking to the notebook so students can watch it work.

    allow_delegation=False,  # this agent may NOT hand its work to another agent.
                             # Delegation is powerful but chatty and expensive; off by default.

    max_iter=3,        # hard cap on think->act loops. Prevents an infinite "let me try again"
                       # spiral. A cost guardrail as much as a safety one.
)

print(f"Agent ready: {classifier_agent.role} | tools: {len(classifier_agent.tools)} "
      f"| max_iter: {classifier_agent.max_iter}")

Agent ready: Ticket Intake Classifier | tools: 0 | max_iter: 3


### Cell 11 — AGENT 2: `research_agent` (the fact-finder)

**What this cell does:** creates the only agent with **read access** to company systems.
It holds `lookup_order` and `check_refund_policy`, and nothing else — notably **not**
`create_escalation`, because a researcher should never take an action.

Watch the `max_iter=6`: this agent genuinely needs several loops (look up order → read result →
check policy → read verdict → summarise), so we give it more room than the classifier. Tuning
`max_iter` per agent is a real skill: too low and it gives up mid-task, too high and a confused
agent burns your budget.

**Input:** the `llm` and the two read-only tools.

**Output:** `Agent ready: Policy & Order Researcher | tools: ['Order Lookup', 'Refund Policy Check'] | max_iter: 6`

In [ ]:
# =============================================================================
# CELL 11 — AGENT 2: THE RESEARCHER  (gathers facts with tools)
# -----------------------------------------------------------------------------
# This is the agent that demonstrates AUTONOMY: we do not tell it "call lookup_order
# then call check_refund_policy". We describe the goal; it picks the tools and the
# order to call them in, at runtime, by itself.
# =============================================================================

research_agent = Agent(
    role="Policy & Order Researcher",

    goal=("Find every fact needed to resolve the ticket: the order record, and the official "
          "refund policy verdict for that order's amount and delivery date."),

    backstory=("You are a meticulous back-office analyst. You NEVER state a fact you did not "
               "read from a tool. You always look up the order first, then feed that order's "
               "amount and delivered_days_ago into the policy checker. You quote the policy "
               "verdict word for word, because the writer downstream depends on it."),

    # THE PERMISSION LIST: two READ tools. No write/action tool is granted here.
    tools=[lookup_order, check_refund_policy],

    llm=llm,
    verbose=True,
    allow_delegation=False,

    max_iter=6,   # needs more loops than the classifier: lookup -> read -> policy -> read -> summarise.
)

print(f"Agent ready: {research_agent.role} | "
      f"tools: {[t.name for t in research_agent.tools]} | max_iter: {research_agent.max_iter}")

Agent ready: Policy & Order Researcher | tools: ['Order Lookup', 'Refund Policy Check'] | max_iter: 6


### Cell 12 — AGENT 3: `writer_agent` (the diplomat)

**What this cell does:** creates the customer-facing agent. It gets exactly one tool —
`create_escalation` — the only *action* tool in the system, held by the only agent that
should ever hand a case to a human.

Look at the backstory: it contains **explicit prohibitions** ("never promise a refund the
policy did not authorise"). Prompt-level rules like this are your *first* line of defence.
They are cheap and helpful — but they are **not sufficient**, because a model can ignore them.
That is why Cell 13 adds a guardrail written in Python, which cannot be talked out of its job.

**Input:** the `llm` and `create_escalation`.

**Output:** `Agent ready: Customer Response Writer | tools: ['Create Human Escalation'] | max_iter: 5`

In [ ]:
# =============================================================================
# CELL 12 — AGENT 3: THE WRITER  (decides and replies)
# -----------------------------------------------------------------------------
# Notice the backstory contains hard rules. That is "soft" guardrailing: helpful,
# but a language model CAN ignore it. Cell 13 adds the "hard" guardrail in code.
# =============================================================================

writer_agent = Agent(
    role="Customer Response Writer",

    goal=("Turn the classification and the researched facts into a short, warm, accurate reply "
          "to the customer, escalating to a human whenever the policy demands it."),

    backstory=("You are a senior support specialist known for calm, human replies. "
               "Your unbreakable rules: (1) never promise a refund that the policy verdict did "
               "not authorise; (2) if the verdict says NEEDS-HUMAN-APPROVAL, you MUST call the "
               "Create Human Escalation tool and tell the customer a manager is reviewing; "
               "(3) never invent delivery dates, ticket numbers or amounts; "
               "(4) keep the reply under 150 words and never mention that you are an AI."),

    # The only ACTION tool in the whole system, given to exactly one agent.
    tools=[create_escalation],

    llm=llm,
    verbose=True,
    allow_delegation=False,

    max_iter=5,
)

print(f"Agent ready: {writer_agent.role} | "
      f"tools: {[t.name for t in writer_agent.tools]} | max_iter: {writer_agent.max_iter}")

Agent ready: Customer Response Writer | tools: ['Create Human Escalation'] | max_iter: 5


### Cell 13 — THE GUARDRAIL: `refund_guardrail`

**What this cell does:** defines a plain Python function that inspects the writer's answer
**before** anyone sees it. CrewAI calls it automatically because we will attach it to the last
task in Cell 14.

**The contract (memorise this):** a guardrail takes the task output and returns a **tuple of two
things** — `(True, cleaned_result)` to let it pass, or `(False, "what went wrong")` to reject it.
On rejection CrewAI **feeds your message back to the agent and makes it try again**, up to
`guardrail_max_retries`. So your error message is not a log line — it is a *correction prompt*.
Write it as an instruction.

**The four checks we enforce:**
1. the answer really is valid JSON matching our schema;
2. no auto-refund above USD 200;
3. if a refund > 200 is involved, `needs_human_escalation` must be `true`;
4. the customer reply is under 150 words.

**Input (at runtime):** the writer task's output object.

**Output:** `(True, <TicketTriage>)` or `(False, "<correction message>")`.

**Output of running the cell:** `Guardrail defined. Auto-refund ceiling = USD 200.0`

In [ ]:
# =============================================================================
# CELL 13 — THE HARD GUARDRAIL (deterministic Python, cannot be argued with)
# -----------------------------------------------------------------------------
# CONTRACT:  guardrail(output) -> (True, value)   ... accept, pass `value` onward
#                              -> (False, "text") ... reject, and RE-PROMPT the agent
#                                                     with "text" as the correction.
# =============================================================================

MAX_AUTO_REFUND_USD = REFUND_POLICY["auto_refund_limit_usd"]   # 200.0
MAX_REPLY_WORDS = 150


def refund_guardrail(output) -> Tuple[bool, Any]:
    """Validate the writer's answer before it is allowed to leave the system."""

    # ---- CHECK 0: get the raw text ------------------------------------------
    # CrewAI hands us a TaskOutput object whose .raw attribute holds the text.
    # getattr(..., fallback) makes this also work with a plain string, which lets
    # us unit-test the guardrail without spending an API call (see the last cell).
    raw_text = getattr(output, "raw", None) or str(output)

    # ---- CHECK 1: is it actually valid JSON in our schema? -------------------
    # Models wrap JSON in ```json fences, add a chatty sentence, and - if they are
    # REASONING models like Qwen3 - emit a <think>...</think> block first. We remove
    # all three wrappers, then find the outermost { ... } and parse that.
    cleaned = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL).strip()
    cleaned = cleaned.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start == -1 or end == -1:
        return (False, "Your answer was not JSON. Reply with ONLY a JSON object matching the "
                       "TicketTriage schema — no prose, no markdown fences.")
    try:
        data = json.loads(cleaned[start:end + 1])
        triage = TicketTriage(**data)          # Pydantic validates types AND required fields.
    except Exception as e:
        return (False, f"Your JSON did not match the required schema ({e}). "
                       f"Return every field: {list(TicketTriage.model_fields.keys())}.")

    # ---- CHECK 2: money ceiling ---------------------------------------------
    # The single most important rule: the machine may never hand out more than USD 200
    # on its own authority, no matter how convincing the customer's email was.
    if triage.refund_amount_usd > MAX_AUTO_REFUND_USD and not triage.needs_human_escalation:
        return (False, f"POLICY VIOLATION: you promised a refund of "
                       f"{triage.refund_amount_usd} USD, which is above the "
                       f"{MAX_AUTO_REFUND_USD} USD automatic limit, without escalating. "
                       f"Set needs_human_escalation=true, call the Create Human Escalation tool, "
                       f"set refund_amount_usd to 0, and rewrite the reply to say a manager is reviewing.")

    # ---- CHECK 3: escalation must be backed by a real ticket ----------------
    if triage.needs_human_escalation and triage.escalation_ticket.lower() in ("none", "", "n/a"):
        return (False, "You marked needs_human_escalation=true but gave no ticket ID. "
                       "Call the Create Human Escalation tool and put the returned ESC-xxx id "
                       "into escalation_ticket.")

    # ---- CHECK 4: brevity ---------------------------------------------------
    words = len(triage.customer_reply.split())
    if words > MAX_REPLY_WORDS:
        return (False, f"Your customer_reply is {words} words. Rewrite it in under "
                       f"{MAX_REPLY_WORDS} words, keeping the same decision.")

    # ---- ALL CHECKS PASSED --------------------------------------------------
    # Returning the validated OBJECT (not the raw text) means everything downstream
    # receives clean, typed data. The guardrail both blocks AND upgrades the output.
    return (True, triage)


print(f"Guardrail defined. Auto-refund ceiling = USD {MAX_AUTO_REFUND_USD}, "
      f"reply limit = {MAX_REPLY_WORDS} words.")

Guardrail defined. Auto-refund ceiling = USD 200.0, reply limit = 150 words.


---
# PART 4 — TASKS and the CREW
---

### Cell 14 — Write the three tasks

**What this cell does:** an *Agent* is a worker; a *Task* is the work order. Every task needs a
`description` (what to do) and an `expected_output` (what "done" looks like).

**Three mechanics to point out in class:**

* **`{ticket_text}` is a placeholder.** We will fill it at run time via
  `crew.kickoff(inputs={"ticket_text": ...})`. This is how one crew handles a thousand different tickets.
* **`context=[classify_task]`** wires the pipeline: the researcher automatically receives the
  classifier's output as background. Context is how information flows between agents.
* **The guardrail and the schema hang off the *last* task** — that is the only output that
  leaves the building, so that is where we inspect.

**Input:** the three agents and the guardrail.

**Output:** `3 tasks defined. Guardrail attached to: write_task (retries allowed: 2)`

In [ ]:
# =============================================================================
# CELL 14 — THE THREE WORK ORDERS
# -----------------------------------------------------------------------------
# description     = the instruction given to the agent
# expected_output = the definition of "done" (the agent is graded against this)
# agent           = who does it
# context         = which earlier tasks' outputs get pasted in as background
# =============================================================================

# --- TASK 1: classify --------------------------------------------------------
# Anything in {curly braces} is a placeholder filled in at kickoff() time.
classify_task = Task(
    description=(
        "Read the customer support email below and classify it.\n\n"
        "--- CUSTOMER EMAIL ---\n{ticket_text}\n--- END EMAIL ---\n\n"
        "Decide: (a) the category (refund, delivery, technical, billing, other), "
        "(b) the urgency (low, medium, high), and (c) the order ID mentioned. "
        "If no order ID appears, write 'unknown'. Do not guess facts."
    ),
    expected_output=(
        "Three labelled lines, exactly:\n"
        "category: <value>\nurgency: <value>\norder_id: <value>"
    ),
    agent=classifier_agent,
)

# --- TASK 2: research --------------------------------------------------------
research_task = Task(
    description=(
        "Using the classification you were given, gather the facts needed to resolve this ticket.\n"
        "1. Call the Order Lookup tool with the order ID.\n"
        "2. Take amount_usd and delivered_days_ago from that result and call the "
        "Refund Policy Check tool with them.\n"
        "3. Report both results. Never state a fact you did not read from a tool."
    ),
    expected_output=(
        "A short factual brief containing: the full order record, and the policy verdict "
        "quoted word for word (AUTO-APPROVE, NEEDS-HUMAN-APPROVAL or DENIED)."
    ),
    agent=research_agent,
    context=[classify_task],   # <-- the classifier's answer is handed to this agent automatically.
)

# --- TASK 3: write the reply (the guarded, structured one) -------------------
write_task = Task(
    description=(
        "Write the final customer reply for this ticket.\n"
        "Follow the policy verdict from the research brief EXACTLY.\n"
        "If the verdict is NEEDS-HUMAN-APPROVAL you MUST call the Create Human Escalation tool "
        "first, then quote the returned ESC ticket id in your answer.\n"
        "Respond with ONLY a JSON object — no markdown fences, no commentary."
    ),
    expected_output=(
        "A single JSON object with the keys: category, urgency, order_id, policy_verdict, "
        "refund_amount_usd, needs_human_escalation, escalation_ticket, customer_reply."
    ),
    agent=writer_agent,
    context=[classify_task, research_task],   # sees BOTH earlier results.

    output_pydantic=TicketTriage,   # force the answer into our Cell 9 schema.
    guardrail=refund_guardrail,     # run our Cell 13 safety check on the result.
    guardrail_max_retries=2,        # if the guardrail rejects, let the agent retry twice.
)

print("3 tasks defined. Guardrail attached to: write_task (retries allowed: 2)")

3 tasks defined. Guardrail attached to: write_task (retries allowed: 2)


### Cell 15 — Assemble the crew

**What this cell does:** binds agents + tasks into one runnable object.

**`Process.sequential`** means: run task 1, then 2, then 3, passing results forward. It is
predictable, cheap and easy to debug — start here for every project.

The alternative, **`Process.hierarchical`**, adds a manager agent that *decides* who works next
and may loop back. We build exactly that in **Part 6 (Cells 21–25)** using these same three
agents, and then measure what it costs — so you can choose on evidence rather than taste.

**Input:** the three agents and three tasks.

**Output:** the roster, plus — importantly — **the model each agent is really holding**:
```
Crew assembled | agents: 3 | tasks: 3 | process: sequential
  Ticket Intake Classifier      -> Qwen/Qwen3-32B:cheapest
  Policy & Order Researcher     -> Qwen/Qwen3-32B:cheapest
  Customer Response Writer      -> Qwen/Qwen3-32B:cheapest
```

**Read those three lines every time.** If they show an old model such as `llama3-8b-8192`,
your agents were built from a stale `llm` object — re-run Cell 4, then Cells 10–12, then this
one (or just *Kernel → Restart & Run All*). This is the fastest way to catch the "I changed the
model but nothing changed" problem before you spend free-tier credits on it.

(CrewAI strips the `openai/` client selector once the object is built, so seeing
`Qwen/Qwen3-32B:cheapest` here is correct — the router URL is already baked into the object.)

In [ ]:
# =============================================================================
# CELL 15 — ASSEMBLE THE CREW (the manager object)
# -----------------------------------------------------------------------------
# The Crew owns the agents and the tasks and is what we actually "run".
# =============================================================================

triage_crew = Crew(
    # The roster. Order here is informational; the task list decides execution order.
    agents=[classifier_agent, research_agent, writer_agent],

    # The pipeline. In a sequential process these run strictly top to bottom.
    tasks=[classify_task, research_task, write_task],

    # sequential = a fixed assembly line (predictable, cheap, easy to debug).
    # hierarchical = a manager agent routes work dynamically (flexible, pricier).
    process=Process.sequential,

    verbose=True,   # stream the whole conversation into the notebook so the class can watch.
)

print(f"Crew assembled | agents: {len(triage_crew.agents)} | "
      f"tasks: {len(triage_crew.tasks)} | process: {triage_crew.process}")

# VERIFY THE MODEL. Each agent captured whatever `llm` was when IT was created, so
# editing CREWAI_MODEL without re-running cells 10-12 leaves stale agents behind.
# Printing it here turns a confusing runtime error into an obvious mismatch.
for agent in triage_crew.agents:
    model_in_use = getattr(agent.llm, "model", agent.llm)
    print(f"  {agent.role:<30} -> {model_in_use}")

Crew assembled | agents: 3 | tasks: 3 | process: Process.sequential
  Ticket Intake Classifier       -> Qwen/Qwen3-32B:cheapest
  Policy & Order Researcher      -> Qwen/Qwen3-32B:cheapest
  Customer Response Writer       -> Qwen/Qwen3-32B:cheapest


---
# PART 5 — Run it
---

### Cell 16 — RUN 1: a small refund that the crew can settle by itself

**What this cell does:** feeds a real customer email into the crew with
`kickoff(inputs=...)` — the dictionary keys must match the `{placeholders}` from Cell 14.

**The email:** Ravi bought USD 180 headphones, delivered 5 days ago, one earcup crackles.
That is inside the 30-day window and under the USD 200 ceiling — so the crew should
**auto-approve** and **not** escalate.

**Input:** `ticket_text` — the raw email string.

**Output:** a long verbose trace (each agent thinking and calling tools), then a final JSON
object. Expect roughly:
```
category: refund | urgency: medium | order_id: A-1001
policy_verdict: AUTO-APPROVE ...
refund_amount_usd: 180.0 | needs_human_escalation: False | escalation_ticket: none
```

⏱ **Takes 20–60 seconds.** Watch the trace — that is the whole lesson happening live.

In [ ]:
# =============================================================================
# CELL 16 — RUN 1: THE HAPPY PATH (auto-approve, no human needed)
# -----------------------------------------------------------------------------
# =============================================================================

# The raw, messy input — exactly what a real inbox looks like.
ticket_1 = """
Hi, I ordered a pair of noise cancelling headphones last week, order A-1001.
They arrived on time but the left earcup crackles badly at any volume.
I've tried resetting them twice. Honestly quite disappointed for the price.
Can I get my money back please?
Thanks, Ravi
"""

# kickoff() starts the whole pipeline. The `inputs` dictionary fills the
# {ticket_text} placeholder we wrote inside classify_task's description.
result_1 = triage_crew.kickoff(inputs={"ticket_text": ticket_1})

print("\n" + "=" * 78)
print("RUN 1 FINISHED")
print("=" * 78)
print(result_1)     # printing a CrewOutput shows the final task's answer.

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9b971ebd-6ddf-40f1-886b-7541b298d11a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Read the customer support email below and classify it.                                                   │
│                                                                                                                 │
│  --- CUSTOMER EMAIL ---                                                                                         │
│                                                                                                                 │
│  Hi, I ordered a pair of noise cancelling headphones last week, order A-1001.                                   │
│  They arrived on time but the left earcup crackles badly at any volume.                                         │
│  I've tried resetting them twice. Honestly quite disappointed for the price.                                    │
│  Can I get my money back please?                                                                                │
│  Thanks, Ravi                                                                                                   │
│                                                                                                                 │
│  --- END EMAIL ---                                                                                              │
│                                                                                                                 │
│  Decide: (a) the category (refund, delivery, technical, billing, other), (b) the urgency (low, medium, high),   │
│  and (c) the order ID mentioned. If no order ID appears, write 'unknown'. Do not guess facts.                   │
│  ID: d999b1e1-2a4a-47ee-98c8-0af081051952                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Read the customer support email below and classify it.                                                   │
│                                                                                                                 │
│  --- CUSTOMER EMAIL ---                                                                                         │
│                                                                                                                 │
│  Hi, I ordered a pair of noise cancelling headphones last week, order A-1001.                                   │
│  They arrived on time but the left earcup crackles badly at any volume.                                         │
│  I've tried resetting them twice. Honestly quite disappointed for the price.                                    │
│  Can I get my money back please?                                                                                │
│  Thanks, Ravi                                                                                                   │
│                                                                                                                 │
│  --- END EMAIL ---                                                                                              │
│                                                                                                                 │
│  Decide: (a) the category (refund, delivery, technical, billing, other), (b) the urgency (low, medium, high),   │
│  and (c) the order ID mentioned. If no order ID appears, write 'unknown'. Do not guess facts.                   │
│  Agent: Ticket Intake Classifier                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 9b971ebd-6ddf-40f1-886b-7541b298d11a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RuntimeError: Agent execution was invoked synchronously from within a running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or `await agent.aexecute_task(...)`) when calling from async code.

### Cell 17 — Inspect what really happened

**What this cell does:** `kickoff()` returns a `CrewOutput` object that is far richer than the
text you just printed. Here we open it up:

* `.pydantic` → the **typed** `TicketTriage` object (this is what your CRM would consume)
* `.tasks_output` → one entry per task, so you can audit each agent separately
* `.token_usage` → tokens spent, i.e. **money**. Show this to your class; agent loops are not free.
* `ESCALATION_LOG` → did the writer actually file a ticket?

**Input:** `result_1` from Cell 16.

**Output:** a tidy field-by-field report. For run 1 you should see
`needs_human_escalation : False` and an empty escalation log.

In [ ]:
# =============================================================================
# CELL 17 — UNPACK THE RESULT OBJECT
# -----------------------------------------------------------------------------
# A CrewOutput is not just text. It carries typed data, per-task history and cost.
# =============================================================================

def show(result):
    """Print a CrewOutput in a readable, teachable way."""

    # --- 1. The typed object. This is the payoff of output_pydantic + guardrail. ---
    print("=" * 78)
    print("STRUCTURED RESULT (result.pydantic)")
    print("=" * 78)
    triage = result.pydantic
    if triage is None:
        print("(no pydantic object — the model returned unparsable text)")
    else:
        for field_name in TicketTriage.model_fields:
            value = getattr(triage, field_name)
            if field_name == "customer_reply":
                print(f"{field_name:>24} :")
                print(textwrap.indent(textwrap.fill(value, 70), " " * 27))
            else:
                print(f"{field_name:>24} : {value}")

    # --- 2. Per-task audit trail: who produced what, in order. ------------------
    print("\n" + "=" * 78)
    print("PER-TASK OUTPUTS (result.tasks_output)")
    print("=" * 78)
    for i, task_output in enumerate(result.tasks_output, start=1):
        preview = task_output.raw.strip().replace("\n", " ")[:160]
        print(f"  Task {i} [{task_output.agent}]: {preview}...")

    # --- 3. Cost. Every agent loop is tokens, and tokens are rupees/dollars. -----
    print("\n" + "=" * 78)
    print("TOKEN USAGE (result.token_usage)  <- this is your bill")
    print("=" * 78)
    print(" ", result.token_usage)

    # --- 4. Side effects: did the writer actually escalate to a human? ----------
    print("\n" + "=" * 78)
    print("ESCALATION AUDIT LOG (real-world side effects)")
    print("=" * 78)
    print(" ", ESCALATION_LOG if ESCALATION_LOG else "(empty — nothing was escalated)")


show(result_1)

### Cell 18 — RUN 2: an expensive refund the crew must **refuse to decide alone**

**What this cell does:** runs a second, harder ticket through the *same* crew — no code changes,
just different input. This is the cell that proves the guardrail earns its keep.

**The email:** Meera's USD 650 monitor, delivered **40 days ago**, and she is threatening legal
action. Two policy problems at once: **outside the 30-day window** *and* **far above the USD 200
ceiling**. A naive chatbot, faced with an angry customer, would very likely just say
*"of course, full refund!"* — that is exactly the failure mode we are engineering against.

**Expected behaviour:**
1. classifier marks `urgency: high` (legal threat),
2. researcher's policy tool returns `DENIED` / `NEEDS-HUMAN-APPROVAL`,
3. writer calls `create_escalation` and gets `ESC-001`,
4. `refund_guardrail` blocks the answer if the writer promised the 650 anyway.

**Input:** `ticket_text` — Meera's email.

**Output:** a trace, then a report with `needs_human_escalation : True`, an
`escalation_ticket : ESC-001`, and a non-empty `ESCALATION_LOG`.

In [ ]:
# =============================================================================
# CELL 18 — RUN 2: THE HARD PATH (must escalate, must not over-promise)
# -----------------------------------------------------------------------------
# SAME crew, SAME code. Only the input changed. That is the value of the design.
# =============================================================================

ticket_2 = """
This is my THIRD email. Order A-1002, the 4K monitor, has dead pixels.
It cost me 650 dollars and it has been over a month of being ignored.
I want a FULL REFUND immediately or I am contacting my lawyer and posting
this everywhere. Do not send me another template response.
- Meera
"""

result_2 = triage_crew.kickoff(inputs={"ticket_text": ticket_2})

print("\n" + "=" * 78)
print("RUN 2 FINISHED")
print("=" * 78)
show(result_2)

### Cell 19 — Prove the guardrail works, without spending an API call

**What this cell does:** in Run 2 the agent probably behaved well, so you never *saw* the
guardrail fire. Here we hand it a deliberately bad answer — the kind a jailbroken or simply
over-eager agent would produce — and watch it get rejected.

This is also how you should **unit-test** guardrails in real projects: no LLM, no cost,
instant feedback, runnable in CI.

**Input:** three hand-written fake answers (good, over-promising, too long).

**Output:**
```
CASE 1 (valid, small refund)   -> PASSED
CASE 2 (650 USD, no escalation)-> BLOCKED: POLICY VIOLATION: you promised a refund of 650.0 USD...
CASE 3 (reply far too long)    -> BLOCKED: Your customer_reply is 240 words...
```

In [ ]:
# =============================================================================
# CELL 19 — UNIT-TEST THE GUARDRAIL (no LLM, no cost, instant)
# -----------------------------------------------------------------------------
# SimpleNamespace(raw=...) builds a tiny fake object with just a .raw attribute,
# which is all our guardrail reads. This is standard testing practice.
# =============================================================================

def fake_output(payload: dict):
    """Wrap a dictionary so it looks like a CrewAI TaskOutput to the guardrail."""
    return SimpleNamespace(raw=json.dumps(payload))


# A reusable template so each case differs by only the interesting field.
base = {
    "category": "refund",
    "urgency": "medium",
    "order_id": "A-1002",
    "policy_verdict": "NEEDS-HUMAN-APPROVAL",
    "refund_amount_usd": 0.0,
    "needs_human_escalation": False,
    "escalation_ticket": "none",
    "customer_reply": "Thanks for writing in. We have started a review and will update you soon.",
}

# CASE 1 — a well-behaved answer: small refund, inside every rule.
case_1 = {**base, "policy_verdict": "AUTO-APPROVE", "refund_amount_usd": 180.0}

# CASE 2 — the dangerous one: promises 650 USD with NO human escalation.
case_2 = {**base, "refund_amount_usd": 650.0, "needs_human_escalation": False}

# CASE 3 — a rambling 200-word reply that breaks the brevity rule.
case_3 = {**base, "customer_reply": "We are very sorry about this. " * 40}

for label, case in [("CASE 1 (valid, small refund)   ", case_1),
                    ("CASE 2 (650 USD, no escalation)", case_2),
                    ("CASE 3 (reply far too long)    ", case_3)]:
    passed, payload = refund_guardrail(fake_output(case))
    if passed:
        print(f"{label} -> PASSED  (guardrail returned a clean {type(payload).__name__})")
    else:
        print(f"{label} -> BLOCKED: {textwrap.shorten(str(payload), 110)}")

### Cell 20 — Print the end-to-end runtime trace

**What this cell does:** prints the exact sequence of calls that just happened, using the real
names from this notebook. **This printout matches the flow slide in the deck one-to-one** — put
them side by side in class so students can connect the diagram to the code.

**Input:** nothing (it is a narrated summary).

**Output:** a numbered call trace from user input to final reply.

In [ ]:
# =============================================================================
# CELL 20 — THE RUNTIME CALL TRACE (what actually happened, in order)
# =============================================================================

TRACE = """
USER INPUT
  raw email text
        |
        v
triage_crew.kickoff(inputs={"ticket_text": ...})        <- Crew (Process.sequential)
        |
  [1] classify_task  ->  classifier_agent  (tools: none, max_iter=3)
        |                    LLM: Qwen/Qwen3-32B:cheapest  (via Hugging Face router)
        |                    out: category / urgency / order_id
        v
  [2] research_task  ->  research_agent    (context = classify_task)
        |                    calls tool  lookup_order(order_id="A-1002")
        |                       -> "Order A-1002 | amount_usd=650.0 | delivered_days_ago=40 ..."
        |                    calls tool  check_refund_policy(amount_usd=650, days_since_delivery=40)
        |                       -> "POLICY VERDICT: DENIED / NEEDS-HUMAN-APPROVAL ..."
        |                    out: factual brief
        v
  [3] write_task     ->  writer_agent      (context = classify_task + research_task)
        |                    calls tool  create_escalation(order_id, reason, amount_usd)
        |                       -> "ESCALATION CREATED: ticket ESC-001 ..."
        |                    drafts JSON matching TicketTriage
        v
  refund_guardrail(output)                              <- deterministic Python
        |  (False, "POLICY VIOLATION: ...")  --> re-prompt writer_agent  (max 2 retries)
        |  (True,  TicketTriage object)      --> continue
        v
  output_pydantic = TicketTriage  (typed, validated)
        |
        v
CrewOutput
  .pydantic      -> TicketTriage object   -> your CRM
  .tasks_output  -> per-agent audit trail -> your logs
  .token_usage   -> tokens spent          -> your bill
  ESCALATION_LOG -> real side effects     -> the human manager's queue
"""

print(TRACE)

---
# PART 6 — The other topology: `Process.hierarchical`

Everything up to here used `Process.sequential`: **you** wrote the order, and the crew followed
it. Task 1, then 2, then 3. Every run identical.

Now we hand that decision to the machine.

In `Process.hierarchical` you add a **manager agent**. You still give the crew a list of tasks,
but you no longer say who does what. The manager reads each task, looks at the roster of
specialists, and **delegates at runtime** — deciding who works, in what order, and whether to go
back and ask someone a follow-up question.

### The one sentence that explains the whole mechanism

> In hierarchical mode, **every task is executed by the manager**, who then delegates to a
> coworker using a tool call — exactly the same tool-calling machinery your `research_agent`
> uses to call `lookup_order`.

That is worth pausing on in class. Delegation is not a special framework feature. The manager is
just an agent whose *tools are the other agents*. CrewAI injects two of them automatically:

| Injected tool | What the manager uses it for |
|---|---|
| `Delegate work to coworker` | "You, Policy & Order Researcher — go do this and report back." |
| `Ask question to coworker` | "Quick question, Researcher: what was the delivery date again?" |

Both address the coworker **by their `role` string**. That has a practical consequence you will
see in the trace: *role names stop being decorative documentation and become addresses.* Two
agents with vague or similar roles will get confused for each other.

### What we build in this part

| Cell | What |
|---|---|
| 21 | `triage_manager` — the manager agent, and the three rules it must obey |
| 22 | The same three tasks, rewritten **without** `agent=` |
| 23 | `hierarchical_crew` — assembled with `manager_agent=` |
| 24 | Run the hard USD 650 ticket through it |
| 25 | Sequential vs hierarchical, measured — tokens, calls and cost |
---

### Cell 21 — Build the manager agent

**What this cell does:** creates the agent that will do the routing. It is an ordinary `Agent`,
but CrewAI enforces **three hard rules** on a manager, and violating them fails at different
times — so learn them now rather than by traceback:

| Rule | What happens if you break it | Where it is enforced |
|---|---|---|
| A manager must have **no tools** | `Exception: Manager agent should not have tools` | at `kickoff()`, inside `_create_manager_agent()` |
| The manager must **not** be in the crew's `agents=[...]` list | `manager_agent_in_agents` validation error | immediately, when you build the `Crew` |
| You must supply `manager_agent=` **or** `manager_llm=` | `missing_manager_llm_or_manager_agent` | immediately, when you build the `Crew` |

The no-tools rule surprises people. The reason is that CrewAI **replaces** the manager's toolset
with the delegation tools — giving it `lookup_order` as well would let it skip its own team and
do the work itself, which defeats the point of having one.

`allow_delegation` is forced to `True` on a manager whether you set it or not, so we set it
explicitly for readability.

**Input:** the shared `llm` from Cell 4.

**Output:** `Manager ready: Support Operations Manager | tools: 0 (delegation tools are injected at kickoff)`

In [ ]:
# =============================================================================
# CELL 21 — THE MANAGER AGENT
# -----------------------------------------------------------------------------
# This agent does no support work. Its entire job is to decide WHO does the work
# and in WHAT ORDER, then hand the pieces out and assemble the result.
# =============================================================================

triage_manager = Agent(
    # The role is doubly important here: the manager addresses its coworkers BY ROLE,
    # and its own role is what shapes how it thinks about supervising.
    role="Support Operations Manager",

    goal=("Resolve each customer ticket correctly and cheaply by delegating every piece of work "
          "to the right specialist, then assembling their findings into one final answer."),

    # The backstory is where we teach it HOW to manage. Without this it tends to either
    # (a) do all the work itself, or (b) delegate the same task repeatedly. Both are common
    # hierarchical failure modes, and both are expensive.
    backstory=(
        "You run a support desk of three specialists and you know exactly what each one is for:\n"
        " - 'Ticket Intake Classifier' reads raw emails and labels them. It has no tools.\n"
        " - 'Policy & Order Researcher' looks up orders and checks refund policy. It has the tools.\n"
        " - 'Customer Response Writer' writes the customer reply and files escalations.\n"
        "Your rules: delegate rather than doing work yourself; never ask the same specialist "
        "the same question twice; always get the facts from the Researcher BEFORE asking the "
        "Writer for a reply, because a reply written without the policy verdict is worthless. "
        "Pass the coworker everything they need in the delegated request - they cannot see your "
        "other conversations. When you have the final reply, return it verbatim and add nothing."
    ),

    # RULE 1: A MANAGER MUST HAVE NO TOOLS.
    # CrewAI injects "Delegate work to coworker" and "Ask question to coworker" at kickoff.
    # Giving it lookup_order here would raise an Exception - and would also tempt it to do
    # the researcher's job itself, which is exactly what we are trying to avoid.
    tools=[],

    llm=llm,               # the same Qwen model. In production, managers often get a BIGGER
                           # model than the workers: routing badly is costlier than writing badly.

    verbose=True,

    allow_delegation=True, # forced True for managers anyway; stated here so the code reads clearly.

    max_iter=12,           # Managers need MANY more loops than workers: every delegation is
                           # an iteration, and it must also read each reply and decide again.
                           # Too low and it gives up mid-plan with a half-finished answer.
)

print(f"Manager ready: {triage_manager.role} | tools: {len(triage_manager.tools)} "
      f"(delegation tools are injected at kickoff)")
print(f"Will supervise: {[a.role for a in [classifier_agent, research_agent, writer_agent]]}")

### Cell 22 — The same three tasks, with `agent=` removed

**What this cell does:** rebuilds our three tasks as `h_classify_task`, `h_research_task` and
`h_write_task` — identical descriptions, one difference: **no `agent=`**.

**Why remove it?** Because in hierarchical mode CrewAI ignores it. Looking at the framework
source, task routing is literally this:

```python
def _get_agent_to_use(self, task):
    if self.process == Process.hierarchical:
        return self.manager_agent     # <- always the manager, whatever the task says
    return task.agent
```

So leaving `agent=classifier_agent` on a hierarchical task is not an error — it is worse, it is
**misleading**. A reader assumes it controls something. Delete it, and the code tells the truth:
*the manager decides.*

**Why fresh objects rather than reusing the originals?** A `Task` stores its result in
`task.output` after a run. Reusing the sequential tasks would mix two runs' state and make the
comparison in Cell 25 meaningless. Fresh objects, clean measurement.

**Note we keep `context=[...]`.** Context is not the same thing as assignment: it declares which
*earlier results* get pasted in as background, which still applies. And we keep the guardrail and
the schema on the last task — safety must not depend on which topology you chose.

**Input:** the guardrail and schema from Cells 9 and 13.

**Output:** `3 hierarchical tasks defined (no agent= on any of them).`

In [ ]:
# =============================================================================
# CELL 22 — TASKS WITHOUT AN ASSIGNED AGENT
# -----------------------------------------------------------------------------
# Same work, but nobody is named. The manager will decide who does each piece.
# =============================================================================

h_classify_task = Task(
    description=(
        "Read the customer support email below and classify it.\n\n"
        "--- CUSTOMER EMAIL ---\n{ticket_text}\n--- END EMAIL ---\n\n"
        "Decide: (a) the category (refund, delivery, technical, billing, other), "
        "(b) the urgency (low, medium, high), and (c) the order ID mentioned. "
        "If no order ID appears, write 'unknown'. Do not guess facts."
    ),
    expected_output=(
        "Three labelled lines, exactly:\n"
        "category: <value>\nurgency: <value>\norder_id: <value>"
    ),
    # NO agent= HERE. In hierarchical mode CrewAI would ignore it anyway;
    # omitting it makes the code honest about who is really choosing.
)

h_research_task = Task(
    description=(
        "Using the classification you were given, gather the facts needed to resolve this ticket.\n"
        "1. Call the Order Lookup tool with the order ID.\n"
        "2. Take amount_usd and delivered_days_ago from that result and call the "
        "Refund Policy Check tool with them.\n"
        "3. Report both results. Never state a fact you did not read from a tool."
    ),
    expected_output=(
        "A short factual brief containing: the full order record, and the policy verdict "
        "quoted word for word (AUTO-APPROVE, NEEDS-HUMAN-APPROVAL or DENIED)."
    ),
    context=[h_classify_task],   # context still matters: it decides what information flows.
)

h_write_task = Task(
    description=(
        "Write the final customer reply for this ticket.\n"
        "Follow the policy verdict from the research brief EXACTLY.\n"
        "If the verdict is NEEDS-HUMAN-APPROVAL you MUST call the Create Human Escalation tool "
        "first, then quote the returned ESC ticket id in your answer.\n"
        "Respond with ONLY a JSON object - no markdown fences, no commentary."
    ),
    expected_output=(
        "A single JSON object with the keys: category, urgency, order_id, policy_verdict, "
        "refund_amount_usd, needs_human_escalation, escalation_ticket, customer_reply."
    ),
    context=[h_classify_task, h_research_task],

    # THE SAFETY LAYER IS IDENTICAL. Changing topology must never change the guarantees:
    # the money ceiling is enforced the same way whichever agent ends up writing the reply.
    output_pydantic=TicketTriage,
    guardrail=refund_guardrail,
    guardrail_max_retries=2,
)

print(f"3 hierarchical tasks defined (no agent= on any of them).")
print("  agent on h_classify_task :", h_classify_task.agent)
print("  agent on h_research_task :", h_research_task.agent)
print("  agent on h_write_task    :", h_write_task.agent)

### Cell 23 — Assemble the hierarchical crew

**What this cell does:** the payoff. Compare the two `Crew(...)` calls side by side — the
difference is three lines:

```python
triage_crew = Crew(                          hierarchical_crew = Crew(
    agents=[classifier, research, writer],       agents=[classifier, research, writer],
    tasks=[classify, research, write],           tasks=[h_classify, h_research, h_write],
    process=Process.sequential,                  process=Process.hierarchical,
)                                                manager_agent=triage_manager,
                                             )
```

The agents are *the same objects*. Only the coordination changed. That is the real lesson of this
section: **topology is a property of the crew, not of the agents.** Your specialists, tools and
guardrails are portable across both.

**Note the manager is deliberately absent from `agents=[...]`.** CrewAI validates this and will
refuse to build the crew otherwise, because the manager is a supervisor, not a member of the pool
it delegates into. (If you prefer not to write a manager yourself, drop `manager_agent=` and pass
`manager_llm=llm` instead — CrewAI generates a generic manager for you. Ours is better, because
its backstory names the specialists.)

**Input:** the three agents, the three hierarchical tasks, and the manager.

**Output:** `Hierarchical crew assembled | workers: 3 | manager: Support Operations Manager | tasks: 3`

In [ ]:
# =============================================================================
# CELL 23 — THE HIERARCHICAL CREW
# -----------------------------------------------------------------------------
# Same agents. Same tools. Same guardrail. Different coordination.
# =============================================================================

hierarchical_crew = Crew(
    # The WORKER POOL. The manager delegates into this list, addressing them by role.
    # NOTE: triage_manager is deliberately NOT here - CrewAI rejects a crew whose manager
    # also appears in `agents`, because a supervisor cannot be its own subordinate.
    agents=[classifier_agent, research_agent, writer_agent],

    tasks=[h_classify_task, h_research_task, h_write_task],

    # THE ONE-LINE CHANGE. Sequential = you decide the order. Hierarchical = the manager does.
    process=Process.hierarchical,

    # REQUIRED for hierarchical: either a manager_agent (ours) or a manager_llm (auto-generated).
    # Omit both and the Crew refuses to build with "missing_manager_llm_or_manager_agent".
    manager_agent=triage_manager,

    verbose=True,
)

print(f"Hierarchical crew assembled | workers: {len(hierarchical_crew.agents)} | "
      f"manager: {hierarchical_crew.manager_agent.role} | tasks: {len(hierarchical_crew.tasks)}")
print(f"process = {hierarchical_crew.process}")
print("\nManager will address these coworkers BY ROLE STRING:")
for a in hierarchical_crew.agents:
    print(f"   '{a.role}'  (tools: {[t.name for t in a.tools] or 'none'})")

### Cell 24 — Run the hard ticket through the manager

**What this cell does:** sends Meera's USD 650 / 40-days-old / legal-threat ticket — the same
input as Cell 18 — through the hierarchical crew, so the two runs are directly comparable.

**What to watch for in the trace.** This is the most interesting output in the notebook; read it
live with the class:

1. The manager thinks about the whole job before acting.
2. It calls `Delegate work to coworker` with `coworker="Ticket Intake Classifier"` — that is a
   **tool call**, printed like any other.
3. The classifier runs as a *sub-agent*, returns its labels, and control comes back to the manager.
4. The manager decides what is needed next. It may delegate to the Researcher, or it may use
   `Ask question to coworker` for a quick clarification instead of a whole task.
5. Eventually it delegates the writing, gets the JSON back, and `refund_guardrail` runs on it —
   **unchanged**. If the writer over-promised, the retry loop works exactly as before.

**Input:** `ticket_text` — the same `ticket_2` from Cell 18.

**Output:** a much longer trace than the sequential run (that is the point), then the same
structured report. Expect `needs_human_escalation: True` and an `ESC-xxx` ticket.

⏱ **Slower and pricier than Cell 18** — typically 1.5–3× the tokens, because every delegation is
an extra round trip carrying its own context. Cell 25 measures exactly how much.

In [ ]:
# =============================================================================
# CELL 24 — RUN THE HIERARCHICAL CREW ON THE SAME HARD TICKET
# -----------------------------------------------------------------------------
# Identical input to Cell 18, so the comparison in Cell 25 is apples-to-apples.
# =============================================================================

# Clear the escalation log so we can see what THIS run files, not what Cell 18 filed.
print("Escalations before this run:", ESCALATION_LOG)
ESCALATION_LOG.clear()

result_h = hierarchical_crew.kickoff(inputs={"ticket_text": ticket_2})

print("\n" + "=" * 78)
print("HIERARCHICAL RUN FINISHED")
print("=" * 78)
show(result_h)     # the same reporting helper we wrote in Cell 17

### Cell 25 — Measure it: sequential vs hierarchical

**What this cell does:** puts the two runs side by side on the numbers that decide which topology
you ship. Opinions about architecture are cheap; `token_usage` is not.

**Input:** `result_2` (sequential, Cell 18) and `result_h` (hierarchical, Cell 24).

**Output:** a comparison table plus a verdict line, e.g.
```
                          SEQUENTIAL   HIERARCHICAL     DELTA
LLM calls                          9             17     +89%
Total tokens                   11,240         21,880    +95%
Escalation filed                 True           True
Same decision?                            YES
```

**How to read it in class.** Both topologies got the right answer — the guardrail saw to that.
The hierarchical run simply paid roughly twice as much for it, because coordinating *is* work:
every delegation is a message that must be written, sent, read and replied to. On a fixed
three-step pipeline that spend buys nothing.

**So when is it worth paying?** When the order genuinely cannot be known in advance:

* the path branches on what an earlier step *found* (a fraud signal changes who reviews next);
* some tickets need three specialists and some need one;
* a step may need to loop back and re-ask;
* you want to add a tenth specialist without rewriting the pipeline.

**And when is it not?** Our case. Classify → research → write, every single time. When you can
draw the flowchart, write the flowchart — `Process.sequential` is cheaper, faster, reproducible
and far easier to debug, because "task 3 failed" points at one agent instead of a conversation.

In [ ]:
# =============================================================================
# CELL 25 — THE COMPARISON THAT DECIDES WHICH TOPOLOGY YOU SHIP
# =============================================================================

def usage_of(result):
    """Pull the numbers out of a CrewOutput's token_usage, whatever its exact shape."""
    u = result.token_usage
    # UsageMetrics is a pydantic object in current CrewAI; older versions returned a dict.
    get = (lambda k: getattr(u, k, 0)) if not isinstance(u, dict) else (lambda k: u.get(k, 0))
    return {
        "calls":  get("successful_requests"),
        "prompt": get("prompt_tokens"),
        "completion": get("completion_tokens"),
        "total":  get("total_tokens"),
    }


def pct(new, old):
    """Percentage change, guarding against a zero baseline."""
    return "n/a" if not old else f"{(new - old) / old * 100:+.0f}%"


seq = usage_of(result_2)     # the sequential run from Cell 18
hier = usage_of(result_h)    # the hierarchical run from Cell 24

print("=" * 74)
print(f"{'':<26}{'SEQUENTIAL':>13}{'HIERARCHICAL':>15}{'DELTA':>10}")
print("=" * 74)
for label, key in [("LLM calls", "calls"), ("Prompt tokens", "prompt"),
                   ("Completion tokens", "completion"), ("Total tokens", "total")]:
    print(f"{label:<26}{seq[key]:>13,}{hier[key]:>15,}{pct(hier[key], seq[key]):>10}")

# Did the extra spend change the ANSWER? Usually not - which is the whole point.
seq_t, hier_t = result_2.pydantic, result_h.pydantic
print("-" * 74)
print(f"{'Tasks executed':<26}{len(result_2.tasks_output):>13}{len(result_h.tasks_output):>15}")
if seq_t and hier_t:
    print(f"{'Escalated to a human':<26}{str(seq_t.needs_human_escalation):>13}"
          f"{str(hier_t.needs_human_escalation):>15}")
    print(f"{'Refund promised (USD)':<26}{seq_t.refund_amount_usd:>13,.2f}"
          f"{hier_t.refund_amount_usd:>15,.2f}")
    same = (seq_t.needs_human_escalation == hier_t.needs_human_escalation
            and seq_t.refund_amount_usd == hier_t.refund_amount_usd)
    print("=" * 74)
    print(f"Same business decision?  {'YES' if same else 'NO'}")
    if same:
        print("Both topologies were right. The hierarchical run just paid more to get there -")
        print("because on a fixed three-step pipeline, coordination buys you nothing.")
else:
    print("(One run did not produce a typed object - check its guardrail output above.)")

print("\nEscalations filed by the hierarchical run:", ESCALATION_LOG)

---
# PART 7 — Recap, gotchas and exercises
---

### The six concepts, and where each one lives in this notebook

| Concept from the deck | Where you saw it |
|---|---|
| **Agent Setup** | Cells 10–12 — `role`, `goal`, `backstory`, `tools`, `llm`, `max_iter` |
| **Architecture** | Cells 14–15 — Tasks + `context` wiring + `Crew` + `Process.sequential` |
| **Roles** | Three narrow specialists instead of one do-everything agent |
| **Autonomy & Focus** | Cell 11 — nobody told the researcher *which* tool to call; `max_iter`, `temperature=0.1` and empty tool lists keep it on-rails |
| **Safety Guardrails** | Cell 7 (rules in code), Cell 12 (rules in prompt), Cell 13 (hard validation), Cell 19 (tests) |
| **Applications** | The whole thing — swap the tools and you have claims triage, KYC review, IT helpdesk, invoice matching |
| **Both topologies** | Cells 14–15 (sequential) vs Cells 21–25 (hierarchical + a manager agent) |

---

### Five gotchas that bite every beginner

1. **Vague tool docstrings.** The agent picks tools by reading the docstring. Vague in, wrong tool out.
2. **Letting the LLM do arithmetic or policy.** Push every threshold into Python (Cell 7).
3. **No `max_iter`.** A confused agent will loop until your budget is gone.
4. **Trusting prompt rules alone.** "Never promise a refund" in a backstory is a *suggestion*. Cell 13 is a *rule*.
5. **Forgetting `context=[...]`.** Without it, agent 2 has no idea what agent 1 concluded.
6. **Hardcoding a model name.** Provider model IDs get renamed and retired, and the spelling is
   fussy. Read the live list from the API (Cell 4) and switch with `use_model("...")`, which
   validates before you spend free-tier credits.
7. **Changing the model but not re-running the agent cells.** Agents hold the `llm` object they
   were built with. Cell 15 prints each agent's real model — check it, or Restart & Run All.

---

### Exercises (try these live in class)

1. **Break it:** in Cell 13, comment out CHECK 2 and re-run Cell 18. Does the agent now promise USD 650?
2. **Add a tool:** write `check_warranty(order_id)` and grant it to `research_agent` only.
3. **Tighten focus:** set `max_iter=1` on `research_agent`. Watch it fail to finish — and understand why.
4. **New role:** add a `qa_agent` that reviews the reply for tone before it is sent (a 4th task).
5. **Starve the manager:** set `max_iter=3` on `triage_manager` (Cell 21) and re-run Cell 24.
   Watch it run out of turns mid-delegation and return a half-finished answer — then explain why
   a manager needs far more iterations than a worker.
6. **New domain:** replace `ORDERS` with insurance claims and the tools with `lookup_policy` /
   `check_coverage`. Notice how little of the agent code changes.

---

### Where to read more
* CrewAI docs — <https://docs.crewai.com/>
* Agents reference — <https://docs.crewai.com/en/concepts/agents>
* Tasks & guardrails — <https://docs.crewai.com/en/concepts/tasks>
* Custom tools — <https://docs.crewai.com/en/concepts/tools>
* Hugging Face access tokens — <https://huggingface.co/settings/tokens>
* Inference Providers (the router) — <https://huggingface.co/docs/inference-providers/index>
* Credits and pricing — <https://huggingface.co/docs/inference-providers/pricing>